In [1]:
from pyspark.sql import functions as F

# ============================================
# ÉTAPE 0 : LECTURE DES DONNÉES
# ============================================
storage_account = "energybigdatastorage"
container_raw = "raw"

path_raw = f"abfss://{container_raw}@{storage_account}.dfs.core.windows.net/energy_data_extracted/archive (3).zip/uk_bank_holidays.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .load(path_raw)

print(f"Nombre de lignes : {df.count()}")
print(f"Nombre de colonnes : {len(df.columns)}")
df.printSchema()
df.show()

In [2]:
# ============================================
# ÉTAPE 1 : SUPPRIMER LES DOUBLONS
# ============================================
nb_avant = df.count()
df = df.dropDuplicates()
nb_apres = df.count()
print(f"Lignes avant : {nb_avant}")
print(f"Lignes après : {nb_apres}")
print(f"Doublons supprimés : {nb_avant - nb_apres}")

In [3]:
# ============================================
# ÉTAPE 2 : VÉRIFICATION DES VALEURS NULLES
# ============================================
print("=== VALEURS NULLES PAR COLONNE ===")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

print("=== VALEURS VIDES EN STRING ===")
df.select([F.count(F.when(F.col(c) == "", c)).alias(c) for c in df.columns]).show()

In [4]:
# ============================================
# ÉTAPE 3 : CORRECTION DE L'ENCODAGE
# ============================================
print("=== AVANT CORRECTION ===")
df.select("Type").show(25, truncate=False)

# Remplacer les "?" par des apostrophes
df = df.withColumn("Type", F.regexp_replace(F.col("Type"), "\\?", "'"))

print("=== APRÈS CORRECTION ===")
df.select("Type").show(25, truncate=False)

In [5]:
# ============================================
# ÉTAPE 4 : SAUVEGARDE DANS PROCESSED
# ============================================
# Renommer la colonne avec espace
df = df.withColumnRenamed("Bank holidays", "Bank_holidays")

path_processed = f"abfss://processed@{storage_account}.dfs.core.windows.net/uk_bank_holidays/"

df.write.format("delta").mode("overwrite").save(path_processed)
print(" uk_bank_holidays sauvegardé dans processed/uk_bank_holidays/")